In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))

import src.db_builder as db

# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: c:\Users\dacil\Desktop\Adalab\proyectos\Proyectos\e-commerce-operations-insights\notebooks


True

In [2]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [3]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="latin-1")

In [4]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time
135349,TRANSFER,4,4,-143.96,57.58,Shipping on time,0,13,Electronics,San Diego,EE. UU.,Mary,5206,Palmer,Consumer,CA,4212 Cozy Corners,92113,3,Footwear,32.695423,-117.099007,Santiago de los Caballeros,RepÃºblica Dominicana,5206,6448,276,6.40,0.10,16144,31.99,-2.50,2,63.98,57.58,-143.96,Caribbean,PENDING,276,13,Under Armour Women's Ignite Slide,31.99,Standard Class,04-09-2015,04-05-2015,02:40
75392,DEBIT,6,4,50.44,193.99,Late delivery,1,48,Water Sports,Caguas,Puerto Rico,Mary,4218,Rodgers,Consumer,PR,4770 Hidden Heath,725,7,Fan Shop,18.210997,-66.370560,El Paso,Estados Unidos,4218,39245,1073,6.00,0.03,97960,199.99,0.26,1,199.99,193.99,50.44,US Center,COMPLETE,1073,48,Pelican Sunstream 100 Kayak,199.99,Standard Class,08-01-2016,07-26-2016,20:55
3256,PAYMENT,4,4,45.24,113.09,Shipping on time,0,18,Men's Footwear,Caguas,Puerto Rico,Mary,10658,Smith,Home Office,PR,3915 Iron Corner,725,4,Apparel,18.206884,-66.370560,Yakarta,Indonesia,10658,21348,403,16.90,0.13,53383,129.99,0.40,1,129.99,113.09,45.24,Southeast Asia,PENDING_PAYMENT,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Standard Class,11-12-2015,11-08-2015,14:48
176396,DEBIT,2,1,-31.49,179.97,Late delivery,1,17,Cleats,Caguas,Puerto Rico,Mary,3373,Willis,Consumer,PR,9749 Red Barn Harbour,725,4,Apparel,18.255585,-66.370590,Bangkok,Tailandia,3373,29815,365,59.99,0.25,74613,59.99,-0.18,4,239.96,179.97,-31.49,Southeast Asia,COMPLETE,365,17,Perfect Fitness Perfect Rip Deck,59.99,First Class,03-13-2016,03-11-2016,05:10
80625,TRANSFER,6,4,-127.54,173.99,Shipping canceled,0,48,Water Sports,Hampton,EE. UU.,Andrea,5772,Smith,Consumer,VA,4353 Velvet Treasure Island,23666,7,Fan Shop,37.047966,-76.399971,Manila,Filipinas,5772,28925,1073,26.00,0.13,72378,199.99,-0.73,1,199.99,173.99,-127.54,Southeast Asia,CANCELED,1073,48,Pelican Sunstream 100 Kayak,199.99,Standard Class,03-04-2016,02-27-2016,05:21


In [5]:
dim_products = df_data[['product_card_id', 'product_name', 'product_price', 
                        'category_id', 'category_name', 'department_id', 
                        'department_name']].drop_duplicates(subset=['product_card_id']).reset_index(drop=True)

In [6]:
dim_products.tail(5)

,product_card_id,product_name,product_price,category_id,category_name,department_id,department_name
113,646,Merrell Women's Grassbow Sport Hiking Shoe,99.99,30,Men's Golf Clubs,6,Outdoors
114,1361,Toys,11.54,74,Toys,7,Fan Shop
115,1073,Pelican Sunstream 100 Kayak,199.99,48,Water Sports,7,Fan Shop
116,1059,Pelican Maverick 100X Kayak,349.99,48,Water Sports,7,Fan Shop
117,1014,O'Brien Men's Neoprene Life Vest,49.98,46,Indoor/Outdoor Games,7,Fan Shop


In [7]:
dim_customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [8]:
dim_customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,Caguas,PR,725,Puerto Rico


In [9]:
dim_stores = df_data[['latitude', 'longitude', 'shipping_mode']].drop_duplicates().reset_index(drop=True)
dim_stores['store_id'] = dim_stores.index + 1
dim_stores = dim_stores[['store_id', 'latitude', 'longitude', 'shipping_mode']]

In [10]:
dim_stores.sample(5)

,store_id,latitude,longitude,shipping_mode
1400,1401,18.240223,-66.370537,Standard Class
331,332,37.344669,-122.035667,Standard Class
6982,6983,41.860199,-87.767471,First Class
19913,19914,18.200125,-66.370628,Second Class
12454,12455,18.241489,-66.370605,Standard Class


In [11]:
fact_sales = pd.merge(df_data, dim_stores, on=['latitude', 'longitude', 'shipping_mode'], how='left')

In [12]:
fact_sales.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time,store_id
162530,DEBIT,2,4,-265.59,331.98,Advance shipping,0,45,Fishing,Buena Park,EE. UU.,Eric,8783,Ramsey,Home Office,CA,6063 Golden Point,90620,7,Fan Shop,33.839504,-118.011467,BerlÃ­n,Alemania,8783,15226,1004,68.0,0.17,38080,399.98,-0.80,1,399.98,331.98,-265.59,Western Europe,COMPLETE,1004,45,Field & Stream Sportsman 16 Gun Fire Safe,399.98,Standard Class,08-13-2015,08-11-2015,05:59,4505
9362,DEBIT,3,4,77.00,220.00,Advance shipping,0,24,Women's Apparel,Los Angeles,EE. UU.,Sandra,10408,Smith,Consumer,CA,8706 Harvest Green,90027,5,Golf,34.125946,-118.291016,Manila,Filipinas,10408,23512,502,30.0,0.12,58823,50.00,0.35,5,250.00,220.00,77.00,Southeast Asia,ON_HOLD,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,12-13-2015,12-10-2015,04:57,4
67361,DEBIT,2,4,36.45,135.00,Advance shipping,0,24,Women's Apparel,Caguas,Puerto Rico,Mary,9085,Anderson,Corporate,PR,4669 Blue Barn Front,725,5,Golf,18.266369,-66.370537,Mougins,Francia,9085,15091,502,15.0,0.10,37766,50.00,0.27,3,150.00,135.00,36.45,Western Europe,COMPLETE,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,08-11-2015,08-09-2015,06:41,12514
118845,TRANSFER,3,2,91.34,260.98,Late delivery,1,43,Camping & Hiking,Chicago,EE. UU.,Mary,8722,Hunt,Consumer,IL,5547 Old Pines,60639,7,Fan Shop,41.894806,-87.765182,Milwaukee,Estados Unidos,8722,38647,957,39.0,0.13,96471,299.98,0.35,1,299.98,260.98,91.34,US Center,PENDING,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,Second Class,07-21-2016,07-18-2016,03:24,15947
165501,PAYMENT,2,1,26.61,85.00,Late delivery,1,24,Women's Apparel,Sacramento,EE. UU.,Virginia,8217,Cantrell,Home Office,CA,3202 Lazy Maze,95828,5,Golf,38.502476,-121.423187,Chelles,Francia,8217,19431,502,15.0,0.15,48556,50.00,0.31,2,100.00,85.00,26.61,Western Europe,PENDING_PAYMENT,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,First Class,10-13-2015,10-11-2015,15:11,8991


In [13]:
fact_sales = fact_sales[[
    'order_item_id', 'order_id', 'customer_id', 'product_card_id', 'store_id',
    'order_date_dateorders',
    'type', 'order_status', 'days_for_shipping_real', 
    'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk',
    'order_city', 'order_country', 'order_region',
    'sales', 'order_item_quantity', 'order_item_product_price', 
    'order_item_discount', 'order_item_discount_rate', 'order_item_total', 
    'order_item_profit_ratio', 'benefit_per_order', 'order_profit_per_order', 'sales_per_customer'
]]
 
print("¡DataFrames del Modelo en Estrella listos en memoria!")

KeyError: "['order_date_dateorders'] not in index"

In [ ]:
USER  = os.getenv("DB_USER")
PASS  = os.getenv("DB_PASSWORD")
HOST  = os.getenv("DB_HOST")
PORT  = os.getenv("DB_PORT", "3306")   # 3306 como valor por defecto
DB_NAME = os.getenv("DB_NAME")
 
# 1️⃣ Conectar al servidor SIN base de datos

engine_server = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}") 
# 2️⃣ Crear la base de datos si no existe

with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print(f"✅ Base de datos '{DB_NAME}' lista")
 
engine_server.dispose()
 
# 3️⃣ Conectar ahora CON la base de datos

engine = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{DB_NAME}")
print(f"✅ Conectado a '{DB_NAME}'")
 

✅ Base de datos 'dataco' lista
✅ Conectado a 'dataco'


In [ ]:
# Carga del DataFrame en MySQL
# Esto crea automáticamente la tabla products en la base de datos
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
'''# Comprobación previa para evitar errores al asignar la clave primaria
dim_products = dim_products.dropna(subset=["Employee_Number"])
dim_products = dim_products.drop_duplicates(subset=["Employee_Number"])'''
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
# Asignación de la clave primaria una vez creada la tabla
with engine.begin() as con:
    con.execute(text("""
        ALTER TABLE products
        MODIFY product_card_id INT NOT NULL"""))
    con.execute(text("""
        ALTER TABLE products
        ADD PRIMARY KEY (product_card_id)"""))
 
print("¡Listo! Base de datos, tabla y clave primaria creadas correctamente.")

¡Listo! Base de datos, tabla y clave primaria creadas correctamente.
